# Controlled Decoding: шаг 5, scorer, обученный при T=0.7

**Зачем.** В `04` при T=0.7, top-p=0.9 CD со scorer'ом, обученным на ответах при T=1, проиграл best-of-K. Есть две гипотезы:
- **(а)** при T=1 CD в основном отсекал блоки, испорченные шумом сэмплирования; низкая температура делает это бесплатно,
  и выбирать по префиксам больше нечего;
- **(б)** scorer видит распределение, на котором не обучался.

Здесь scorer обучается тем же рецептом, что в `02`, но на роллаутах π_ref при T=0.7, top-p=0.9, и CD при T=0.7 повторяется
с новым scorer'ом. Если CD вернёт преимущество над best-of-K, верна (б). Если нет, верна (а).

| Раздел | Что делает | GPU | Время (2×T4) |
|---|---|---|---|
| 1 | Роллауты: те же 10 000 train-промптов × 2 ответа при T=0.7, reward по Skywork 0.6B | обе | ~40 мин |
| 2 | Обучение scorer'а, рецепт и гиперпараметры как в `02` | одна | ~35 мин |
| 3 | Строгая pair_acc@t: 2 scorer'а × ответы при T=1 и T=0.7 на test | обе | ~15 мин |
| 4 | CD при T=0.7 с новым scorer'ом, те же 3 конфигурации и сиды, что в `04` | обе | ~1 ч |

**Входы (Add Input):** `cd-hh-rollouts`, `cd-hh-scorer`, `cd-hh-eval`, `cd-hh-feedback`. Best-of-K при T=0.7 и CD со старым
scorer'ом берутся из `cd-hh-feedback` и заново не считаются.

**Промежуточные сохранения:** роллауты шардами по 1000 промптов, обучение чекпоинтами, CD шардами по 50 промптов.
Restart и Run All продолжают с места остановки. Для новой сессии подключи датасеты, которые загружает последняя ячейка
(`cd-hh-t07` и `cd-hh-scorer-t07`): готовое подтянется из `/kaggle/input`.

In [ ]:
import os, re, json, glob, time, shutil, math
from collections import defaultdict
from itertools import combinations
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import (AutoConfig, AutoTokenizer, AutoModelForCausalLM, AutoModelForTokenClassification,
                          AutoModelForSequenceClassification, get_cosine_schedule_with_warmup)

print(torch.__version__, [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
DEBUG = True  # 64 train-промпта, 16 test-промптов; для полного прогона поставить False

TEMP_KW = {"temperature": 0.7, "top_p": 0.9}  # как в 04
N_TRAIN = 64 if DEBUG else 10_000             # те же train-промпты, что в 01
N_SAMPLES = 2
ROLL_SHARD = 32 if DEBUG else 1000
GEN_BATCH = 64                                # промптов за вызов generate в роллаутах, как в 01

N_PROMPTS = 16 if DEBUG else 500              # test-промпты
TEMP_CD = [(16, 4), (32, 4), (16, 8)]         # (M, K), как в 04
SEED_04 = 1                                   # сиды CD как в 04: первый блок у старого и нового scorer'а совпадает
PAIR_TS = [8, 16, 32, 64, 128]
PAIR_MIN_GAP = 0.5
SHARD_SIZE = 8 if DEBUG else 50
GEN_SEQS = 128
SCORER_BATCH = 32
RM_BATCH = 32
N_BOOT = 5000
SEED = 0

# Обучение: всё как в 02
TRAIN_HP = dict(val_frac=0.05, epochs=1, micro_bs=8, accum=4, lr=1e-5, head_lr=1e-4, warmup=2 if DEBUG else 30,
                max_grad_norm=1.0, eval_every=5 if DEBUG else 100, save_every=5 if DEBUG else 100, log_every=10,
                eval_prefixes=[8, 16, 32, 64, 128, 256], seed=0)

GEN_DEVICE = "cuda:0"
AUX_DEVICE = "cuda:1" if torch.cuda.device_count() > 1 else "cuda:0"
DEVS = [GEN_DEVICE, AUX_DEVICE] if AUX_DEVICE != GEN_DEVICE else [GEN_DEVICE]
DTYPE = torch.float16

WORK = "/kaggle/working" if os.path.exists("/kaggle") else "."
OUT = f"{WORK}/cd_t07_debug" if DEBUG else f"{WORK}/cd_t07"
RES = f"{OUT}/results"             # плоская папка, уходит в датасет cd-hh-t07
ROLL_SHARDS = f"{OUT}/roll_shards"
SC_OUT = f"{OUT}/scorer_t07"       # ckpt, final, metrics.jsonl; final уходит в датасет cd-hh-scorer-t07
for d in [RES, ROLL_SHARDS, SC_OUT]:
    os.makedirs(d, exist_ok=True)

## Входы

In [ ]:
def find_one(pattern, what):
    found = [f for f in sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)) + sorted(glob.glob(pattern))
             if "debug" not in f]
    assert len(found) == 1, f"{what}: нужен ровно один {pattern}, найдено {found}"
    return os.path.dirname(found[0])

def find_dirs(pattern, ok):
    # папки, где лежит файл pattern и json в нём проходит проверку ok
    return [os.path.dirname(f) for f in sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)) if ok(json.load(open(f)))]

ROLLOUTS = find_one("test_prompts.parquet", "роллауты")
rollout_cfg = json.load(open(f"{ROLLOUTS}/config.json"))
assert not rollout_cfg["debug"], "подключены отладочные роллауты"

old_sc = find_dirs("scorer_config.json", lambda c: c["rollouts_cfg"] == rollout_cfg and not c["train_meta"]["debug"])
assert len(old_sc) == 1, f"нужен ровно один scorer, обученный при T=1 (cd-hh-scorer), найдено {old_sc}"
SCORER_OLD = old_sc[0]

ev = find_dirs("run_meta.json", lambda m: m.get("n_prompts") == 500 and m.get("rollouts_cfg") == rollout_cfg)
ev = [d for d in ev if glob.glob(f"{d}/bon_*.parquet")]
assert ev, "не найдены шарды полного прогона 03: подключи cd-hh-eval"
EVAL = ev[0]

fb = find_dirs("feedback_meta.json", lambda m: m["n_prompts"] == 500 and m["temp_kw"] == TEMP_KW and m["temp_cd"] == [list(x) for x in TEMP_CD])
fb = [d for d in fb if glob.glob(f"{d}/tbon_*.parquet")]
assert fb, "не найдены результаты 04 при T=0.7: подключи cd-hh-feedback"
FEEDBACK = fb[0]
assert json.load(open(f"{FEEDBACK}/feedback_meta.json"))["seed"] == SEED_04

GEN_KW = {k: v for k, v in rollout_cfg["gen_kw"].items() if k != "max_new_tokens"}
MAX_NEW_TOKENS = rollout_cfg["gen_kw"]["max_new_tokens"]
GEN_KW_T07 = GEN_KW | TEMP_KW                     # ровно то, что использовалось в 04
ROLL_CFG_T07 = rollout_cfg | {"gen_kw": rollout_cfg["gen_kw"] | TEMP_KW, "debug": DEBUG, "source": "05_temp07",
                              "n_train_prompts": N_TRAIN}
print("rollouts:", ROLLOUTS, "\nscorer T=1:", SCORER_OLD, "\neval:", EVAL, "\nfeedback:", FEEDBACK)
print("генерация:", GEN_KW_T07)

# Защита от чужих результатов и подтягивание результатов прошлой сессии
T07_META = json.loads(json.dumps({"roll_cfg": ROLL_CFG_T07, "train_hp": TRAIN_HP, "n_prompts": N_PROMPTS, "temp_cd": TEMP_CD,
                                  "seed_04": SEED_04, "shard_size": SHARD_SIZE, "seed": SEED}))
meta_path = f"{RES}/t07_meta.json"
if os.path.exists(meta_path):
    assert json.load(open(meta_path)) == T07_META, f"В {OUT} результаты прогона с другими параметрами. Удали папку: !rm -rf {OUT}"
elif glob.glob(f"{RES}/*.parquet"):
    raise RuntimeError(f"В {RES} файлы неизвестного прогона. Удали папку: !rm -rf {OUT}")
else:
    json.dump(T07_META, open(meta_path, "w"), indent=2)
for prev in find_dirs("t07_meta.json", lambda m: m == T07_META):
    n = 0
    for f in glob.glob(f"{prev}/*"):
        if f.endswith((".parquet", ".json", ".csv")) and not os.path.exists(f"{RES}/{os.path.basename(f)}"):
            shutil.copy(f, RES); n += 1
    print(f"подтянул {n} файлов из {prev}")

test = pd.read_parquet(f"{ROLLOUTS}/test_prompts.parquet").head(N_PROMPTS)
test["prompt_ids"] = test.prompt_ids.map(lambda x: x.tolist())
PIDS = np.sort(test.pid.values)
n_shards = (len(test) + SHARD_SIZE - 1) // SHARD_SIZE

## Общие функции

Те же, что в `03` и `04`. Паддинг сделан вручную, без токенизатора: генерация в разделе 1 идёт из двух потоков.

In [ ]:
tok = AutoTokenizer.from_pretrained(rollout_cfg["base_model"])
tok.padding_side = "left"
rm_tok = AutoTokenizer.from_pretrained(rollout_cfg["rm_model"])
rm_tok.padding_side = "right"

def load_policy(dev):
    p = AutoModelForCausalLM.from_pretrained(rollout_cfg["base_model"], dtype=DTYPE).to(dev).eval()
    eos = p.generation_config.eos_token_id
    assert set(eos if isinstance(eos, list) else [eos]) == EOS_IDS, eos
    return p

def load_rm(dev):
    m = AutoModelForSequenceClassification.from_pretrained(rollout_cfg["rm_model"], dtype=DTYPE, num_labels=1).to(dev).eval()
    m.config.pad_token_id = rm_tok.pad_token_id
    return m

def load_scorer(path, dev):
    m = AutoModelForTokenClassification.from_pretrained(path, dtype=DTYPE).to(dev).eval()
    m.config.pad_token_id = tok.pad_token_id
    return m

EOS_IDS = {151645, 151643}  # <|im_end|>, <|endoftext|>; сверяется с generation_config ниже

def cut_at_eos(ids):
    for i, t in enumerate(ids):
        if t in EOS_IDS:
            return ids[: i + 1], True
    return ids, False

def left_pad(seqs):
    L = max(len(s) for s in seqs)
    ids = torch.full((len(seqs), L), tok.pad_token_id, dtype=torch.long)
    att = torch.zeros((len(seqs), L), dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, L - len(s):] = torch.tensor(s); att[i, L - len(s):] = 1
    return ids, att

def right_pad(seqs):
    L = max(len(s) for s in seqs)
    ids = torch.full((len(seqs), L), tok.pad_token_id, dtype=torch.long)
    att = torch.zeros((len(seqs), L), dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, :len(s)] = torch.tensor(s); att[i, :len(s)] = 1
    return ids, att

@torch.no_grad()
def sample(pol, seqs, n, max_new, gen_kw):
    dev = next(pol.parameters()).device
    out = []
    per_call = max(1, GEN_SEQS // n)
    for b in range(0, len(seqs), per_call):
        ids, att = left_pad(seqs[b:b + per_call])
        g = pol.generate(input_ids=ids.to(dev), attention_mask=att.to(dev), **gen_kw, max_new_tokens=max_new,
                         num_return_sequences=n, pad_token_id=tok.pad_token_id)
        out += [cut_at_eos(o) for o in g[:, ids.shape[1]:].tolist()]
    return out

@torch.no_grad()
def values(sc, seqs):
    # V(x, y) на последнем токене каждой последовательности
    dev = next(sc.parameters()).device
    res = np.empty(len(seqs))
    order = sorted(range(len(seqs)), key=lambda i: len(seqs[i]))
    for k in range(0, len(order), SCORER_BATCH):
        idx = order[k:k + SCORER_BATCH]
        ids, att = right_pad([seqs[i] for i in idx])
        v = sc(input_ids=ids.to(dev), attention_mask=att.to(dev)).logits[..., 0].float()
        last = att.sum(1).to(dev) - 1
        res[idx] = v[torch.arange(len(idx), device=dev), last].cpu().numpy()
    return res

@torch.no_grad()
def values_all(sc, prompts, responses, desc):
    # V после 0..T токенов ответа
    dev = next(sc.parameters()).device
    seqs = [list(p) + list(r) for p, r in zip(prompts, responses)]
    out = [None] * len(seqs)
    order = sorted(range(len(seqs)), key=lambda i: len(seqs[i]))
    for k in tqdm(range(0, len(order), SCORER_BATCH), desc=desc):
        idx = order[k:k + SCORER_BATCH]
        ids, att = right_pad([seqs[i] for i in idx])
        v = sc(input_ids=ids.to(dev), attention_mask=att.to(dev)).logits[..., 0].float().cpu().numpy()
        for r, i in enumerate(idx):
            out[i] = v[r, len(prompts[i]) - 1:len(seqs[i])]
    return out

def chat_text(tk, messages, response):
    t = tk.apply_chat_template(json.loads(messages) + [{"role": "assistant", "content": response}], tokenize=False)
    return t[len(tk.bos_token):] if tk.bos_token and t.startswith(tk.bos_token) else t

@torch.no_grad()
def rewards(rm, messages, responses, bs=RM_BATCH, pbar=None):
    dev = next(rm.parameters()).device
    texts = [chat_text(rm_tok, m, r) for m, r in zip(messages, responses)]
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]))
    res = np.empty(len(texts))
    for b in range(0, len(order), bs):
        idx = order[b:b + bs]
        enc = rm_tok([texts[i] for i in idx], return_tensors="pt", padding=True).to(dev)
        res[idx] = rm(**enc).logits[:, 0].float().cpu().numpy()
        if pbar is not None:
            pbar.update(len(idx))
    assert np.isfinite(res).all(), "inf/nan в reward: проблема с fp16"
    return res

def decode(ids):
    return tok.decode(ids, skip_special_tokens=True).strip()

def save_parquet(df, path):
    # запись через временный файл: при падении сессии посреди записи не останется битого шарда
    df.to_parquet(path + ".tmp")
    os.replace(path + ".tmp", path)

def res_path(name, s):
    return f"{RES}/{name}_{s:03d}.parquet"

def load_shards(d, name):
    files = sorted(glob.glob(f"{d}/{name}_[0-9][0-9][0-9].parquet"))
    if not files:
        return None
    df = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
    df = df[df.pid.isin(PIDS)]
    return df.sort_values(["pid", "sample"] if "sample" in df else "pid").reset_index(drop=True)

## 1. Роллауты при T=0.7

Те же train-промпты, что в `01`, по 2 ответа. Генерация идёт на обеих GPU одновременно: у каждой своя копия π_ref и своя
половина шардов. Сид задаётся для каждого шарда отдельно, поэтому результат не зависит от того, на какой GPU попал шард.

In [ ]:
TRAIN_T07 = f"{RES}/train_t07.parquet"

def gen_path(s):
    return f"{ROLL_SHARDS}/gen_{s:03d}.parquet"

if not os.path.exists(TRAIN_T07):
    tr1 = pd.read_parquet(f"{ROLLOUTS}/train.parquet")
    prompts = tr1.drop_duplicates("pid").sort_values("pid")[["pid", "messages", "prompt_ids"]].head(N_TRAIN).reset_index(drop=True)
    prompts["prompt_ids"] = prompts.prompt_ids.map(lambda x: x.tolist())
    assert len(prompts) == N_TRAIN, f"в train.parquet только {len(prompts)} промптов"
    n_roll = (len(prompts) + ROLL_SHARD - 1) // ROLL_SHARD
    T1_STATS = {"reward": tr1.reward.mean(), "finished": tr1.finished.mean(), "len": tr1.response_ids.map(len).mean(),
                "std": tr1.reward.std(), "std_within": tr1.groupby("pid").reward.std().mean()}
    del tr1

    pols = [load_policy(d) for d in DEVS]  # грузим в основном потоке (см. 04: from_pretrained в потоках ломает dtype)

    @torch.no_grad()
    def gen_worker(w, pbar):
        pol, dev = pols[w], DEVS[w]
        for s in range(w, n_roll, len(DEVS)):
            if os.path.exists(gen_path(s)):
                continue
            torch.cuda.default_generators[torch.device(dev).index].manual_seed(SEED * 100_003 + 70_000 + s)
            shard = prompts.iloc[s * ROLL_SHARD:(s + 1) * ROLL_SHARD].sort_values("prompt_ids", key=lambda c: c.map(len))
            rows = []
            for b in range(0, len(shard), GEN_BATCH):
                chunk = shard.iloc[b:b + GEN_BATCH]
                res = sample(pol, chunk.prompt_ids.tolist(), N_SAMPLES, MAX_NEW_TOKENS, GEN_KW_T07)
                for j, p in enumerate(chunk.itertuples()):
                    for k in range(N_SAMPLES):
                        ids, fin = res[j * N_SAMPLES + k]
                        rows.append({"pid": p.pid, "sample": k, "messages": p.messages, "prompt_ids": p.prompt_ids,
                                     "response_ids": ids, "finished": fin})
                pbar.update(len(chunk))
            save_parquet(pd.DataFrame(rows), gen_path(s))

    done = sum(min(ROLL_SHARD, len(prompts) - s * ROLL_SHARD) for s in range(n_roll) if os.path.exists(gen_path(s)))
    pbar = tqdm(total=len(prompts), initial=done, desc="rollouts T=0.7", unit="prompt")
    with ThreadPoolExecutor(len(DEVS)) as ex:
        for f in [ex.submit(gen_worker, w, pbar) for w in range(len(DEVS))]:
            f.result()
    pbar.close()
    del pols
    torch.cuda.empty_cache()

    # reward по Skywork 0.6B, как в 01
    rm = load_rm(AUX_DEVICE)
    pbar = tqdm(total=N_SAMPLES * len(prompts), desc="reward", unit="resp")
    for s in range(n_roll):
        sp = f"{ROLL_SHARDS}/scored_{s:03d}.parquet"
        if os.path.exists(sp):
            pbar.update(len(pd.read_parquet(sp, columns=["pid"]))); continue
        df = pd.read_parquet(gen_path(s))
        df["response"] = [decode(r) for r in df.response_ids]
        df["reward"] = rewards(rm, df.messages, df.response, pbar=pbar)
        save_parquet(df, sp)
    pbar.close()
    del rm
    torch.cuda.empty_cache()

    train07 = pd.concat([pd.read_parquet(f"{ROLL_SHARDS}/scored_{s:03d}.parquet") for s in range(n_roll)], ignore_index=True)
    save_parquet(train07, TRAIN_T07)
    json.dump(T1_STATS, open(f"{RES}/t1_train_stats.json", "w"))

train07 = pd.read_parquet(TRAIN_T07)
t1s = json.load(open(f"{RES}/t1_train_stats.json"))
t7s = {"reward": train07.reward.mean(), "finished": train07.finished.mean(), "len": train07.response_ids.map(len).mean(),
       "std": train07.reward.std(), "std_within": train07.groupby("pid").reward.std().mean()}
print(pd.DataFrame({"T=1 (01)": t1s, "T=0.7": t7s}).round(3).to_string())
print("\nstd_within: разброс reward между двумя ответами на один промпт. Чем он меньше, тем меньше выигрыш от любого выбора.")

## 2. Обучение scorer'а

Код и гиперпараметры из `02`, отличаются только данные. Чекпоинт каждые 100 шагов в `scorer_t07/ckpt`.

In [ ]:
FINAL = f"{SC_OUT}/final"
CKPT = f"{SC_OUT}/ckpt"
METRICS = f"{SC_OUT}/metrics.jsonl"
INIT_MODEL = rollout_cfg["base_model"]

def scorer_ready():
    # готовый scorer: в этой сессии или в подключённом датасете cd-hh-scorer-t07
    if os.path.exists(f"{FINAL}/scorer_config.json"):
        return FINAL
    c = find_dirs("scorer_config.json", lambda c: c["rollouts_cfg"] == ROLL_CFG_T07 and c["train_meta"]["hp"] == TRAIN_HP)
    return c[0] if c else None

def train_scorer():
    hp = TRAIN_HP
    df = train07
    pids = np.random.default_rng(hp["seed"]).permutation(np.sort(df.pid.unique()))
    val_pids = set(pids[: max(1, int(len(pids) * hp["val_frac"]))].tolist())
    is_val = df.pid.isin(val_pids)
    r_mean, r_std = float(df.reward[~is_val].mean()), float(df.reward[~is_val].std())

    def to_examples(d):
        return [{"ids": np.concatenate([r.prompt_ids, r.response_ids]).astype(np.int64), "p": len(r.prompt_ids),
                 "target": (r.reward - r_mean) / r_std, "reward": r.reward, "pid": r.pid} for r in d.itertuples()]

    train_ex, val_ex = to_examples(df[~is_val]), to_examples(df[is_val])
    print(f"train {len(train_ex)} seq, val {len(val_ex)} seq; reward mean {r_mean:.2f}, std {r_std:.2f}")

    def collate(items):
        L = max(len(x["ids"]) for x in items)
        ids = torch.full((len(items), L), tok.pad_token_id, dtype=torch.long)
        att = torch.zeros((len(items), L), dtype=torch.long)
        mask = torch.zeros((len(items), L), dtype=torch.bool)
        for i, x in enumerate(items):
            n = len(x["ids"])
            ids[i, :n] = torch.from_numpy(x["ids"]); att[i, :n] = 1
            mask[i, x["p"] - 1:n] = True
        return {"input_ids": ids, "attention_mask": att, "mask": mask,
                "target": torch.tensor([x["target"] for x in items], dtype=torch.float32)}

    lengths = [len(x["ids"]) for x in train_ex]
    schedule = []
    for ep in range(hp["epochs"]):
        rng = np.random.default_rng(hp["seed"] + ep)
        idx = rng.permutation(len(train_ex))
        batches = []
        for i in range(0, len(idx), hp["micro_bs"] * 64):
            chunk = sorted(idx[i:i + hp["micro_bs"] * 64], key=lambda j: lengths[j])
            batches += [chunk[k:k + hp["micro_bs"]] for k in range(0, len(chunk), hp["micro_bs"])]
        schedule += [batches[i] for i in rng.permutation(len(batches))]
    total = len(schedule) // hp["accum"]
    print("optimizer steps:", total)

    dev = GEN_DEVICE
    config = AutoConfig.from_pretrained(INIT_MODEL, num_labels=1)
    config.classifier_dropout = 0.0
    model = AutoModelForTokenClassification.from_pretrained(INIT_MODEL, config=config, dtype=torch.float32)
    for m in model.modules():
        if isinstance(m, torch.nn.Dropout):
            m.p = 0.0
    with torch.no_grad():
        for n_, p in model.named_parameters():
            if n_.startswith("score"):
                p.zero_()
    model.config.pad_token_id = tok.pad_token_id
    model.config.use_cache = False
    model.gradient_checkpointing_enable()
    model.to(dev)
    head = [p for n_, p in model.named_parameters() if n_.startswith("score")]
    body = [p for n_, p in model.named_parameters() if not n_.startswith("score")]
    opt = torch.optim.AdamW([{"params": body, "lr": hp["lr"]}, {"params": head, "lr": hp["head_lr"]}], weight_decay=0.0)
    sched = get_cosine_schedule_with_warmup(opt, hp["warmup"], total)
    scaler = torch.amp.GradScaler("cuda", init_scale=2 ** 10)
    meta = {"init_model": INIT_MODEL, "rollouts_cfg": ROLL_CFG_T07, "debug": DEBUG, "hp": hp,
            "n_train": len(train_ex), "n_val": len(val_ex)}

    def save_ckpt(step):
        tmp = CKPT + "_tmp"
        shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp)
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                    "scaler": scaler.state_dict(), "step": step, "meta": meta}, f"{tmp}/state.pt")
        shutil.rmtree(CKPT, ignore_errors=True); os.rename(tmp, CKPT)

    start = 0
    if os.path.exists(f"{CKPT}/state.pt"):
        st = torch.load(f"{CKPT}/state.pt", map_location=dev, weights_only=False)
        assert st["meta"] == meta, "чекпоинт от прогона с другими параметрами"
        model.load_state_dict(st["model"]); opt.load_state_dict(st["opt"]); sched.load_state_dict(st["sched"]); scaler.load_state_dict(st["scaler"])
        start = st["step"]; del st
        print(f"продолжаю с шага {start}/{total}")
    elif os.path.exists(METRICS):
        os.remove(METRICS)

    def log(rec):
        with open(METRICS, "a") as f:
            f.write(json.dumps(rec) + "\n")

    by_pid = defaultdict(list)
    for i, x in enumerate(val_ex):
        by_pid[x["pid"]].append(i)
    val_pairs = [(a, b) for ids in by_pid.values() for a, b in combinations(ids, 2)
                 if abs(val_ex[a]["reward"] - val_ex[b]["reward"]) > PAIR_MIN_GAP]

    @torch.no_grad()
    def evaluate():
        model.eval()
        vals = [None] * len(val_ex)
        order = sorted(range(len(val_ex)), key=lambda i: len(val_ex[i]["ids"]))
        for k in range(0, len(order), 16):
            idxs = order[k:k + 16]
            b = collate([val_ex[i] for i in idxs])
            with torch.autocast("cuda", dtype=torch.float16):
                v = model(input_ids=b["input_ids"].to(dev), attention_mask=b["attention_mask"].to(dev)).logits.squeeze(-1).float().cpu().numpy()
            for r, i in enumerate(idxs):
                vals[i] = v[r, val_ex[i]["p"] - 1:len(val_ex[i]["ids"])]
        model.train()
        tg = np.array([x["target"] for x in val_ex])
        res = {"val_mse": float(np.mean(np.concatenate([(v - t) ** 2 for v, t in zip(vals, tg)]))),
               "corr@full": float(np.corrcoef([v[-1] for v in vals], tg)[0, 1])}
        for t in hp["eval_prefixes"]:  # строгая версия: только пары, где оба ответа длиннее t
            ok = [(vals[a][t] > vals[b][t]) == (tg[a] > tg[b]) for a, b in val_pairs if len(vals[a]) > t + 1 and len(vals[b]) > t + 1]
            res[f"pair_acc@{t}"] = float(np.mean(ok)) if ok else float("nan")
        res["pair_acc@full"] = float(np.mean([(vals[a][-1] > vals[b][-1]) == (tg[a] > tg[b]) for a, b in val_pairs]))
        return res

    fmt = lambda r: " ".join(f"{k}={v:.3f}" for k, v in r.items())
    model.train()
    pbar = tqdm(total=total, initial=start, desc="train", unit="step")
    ema, t0 = None, time.time()
    for step in range(start, total):
        loss_acc = 0.0
        for m in range(hp["accum"]):
            b = {k: v.to(dev) for k, v in collate([train_ex[j] for j in schedule[step * hp["accum"] + m]]).items()}
            with torch.autocast("cuda", dtype=torch.float16):
                v = model(input_ids=b["input_ids"], attention_mask=b["attention_mask"]).logits.squeeze(-1)
            loss = ((v.float() - b["target"][:, None]) ** 2)[b["mask"]].mean()
            scaler.scale(loss / hp["accum"]).backward()
            loss_acc += loss.item() / hp["accum"]
        scaler.unscale_(opt)
        gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), hp["max_grad_norm"]).item()
        scale = scaler.get_scale()
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        if scaler.get_scale() >= scale:
            sched.step()
        ema = loss_acc if ema is None else 0.95 * ema + 0.05 * loss_acc
        pbar.update(1); pbar.set_postfix(loss=f"{ema:.3f}", gnorm=f"{gnorm:.2f}")
        if (step + 1) % hp["log_every"] == 0:
            log({"step": step + 1, "train_loss": loss_acc, "gnorm": gnorm, "sec": time.time() - t0})
        if (step + 1) % hp["eval_every"] == 0 or step + 1 == total:
            r = evaluate(); log({"step": step + 1, **r}); pbar.write(f"step {step + 1}: {fmt(r)}")
        if (step + 1) % hp["save_every"] == 0 or step + 1 == total:
            save_ckpt(step + 1)
    pbar.close()

    final_res = evaluate()
    model.to(torch.float16).save_pretrained(FINAL)
    tok.save_pretrained(FINAL)
    json.dump({"init_model": INIT_MODEL, "reward_mean": r_mean, "reward_std": r_std, "rollouts_cfg": ROLL_CFG_T07,
               "value_position": "logits[P-1+t] = V(x, y^t), P = len(prompt_ids), t = число токенов ответа",
               "train_meta": meta, "val": final_res}, open(f"{FINAL}/scorer_config.json", "w"), indent=2)
    shutil.copy(METRICS, FINAL)
    shutil.rmtree(CKPT, ignore_errors=True)  # 6 ГБ с оптимизатором; финальные веса уже сохранены
    print("final:", fmt(final_res))
    del model, opt
    torch.cuda.empty_cache()

In [ ]:
if scorer_ready() is None:
    train_scorer()
SCORER_NEW = scorer_ready()
print("scorer T=0.7:", SCORER_NEW)
print("val (строгая pair_acc):", json.dumps(json.load(open(f"{SCORER_NEW}/scorer_config.json"))["val"], indent=1))

## 3. Строгая pair_acc@t: какой scorer лучше различает ответы при T=0.7

Пары ответов на test-промптах: сэмплы (0,1), (2,3), …, (30,31) из `03` (T=1) и из `04` (T=0.7). Считаются только пары,
где оба ответа длиннее t, как в `04`. Таблица 2×2 разделяет гипотезы:
- если на ответах при T=0.7 новый scorer заметно точнее старого, старый действительно страдал от сдвига распределения;
- если оба scorer'а одинаково слабы на T=0.7, при низкой температуре ответы одного промпта просто труднее различить.

In [ ]:
def pair_acc_table(vals, lens, rew, pairs):
    rows = []
    for t in PAIR_TS + ["full"]:
        if t == "full":
            ok = [(vals[a][-1] > vals[b][-1]) == (rew[a] > rew[b]) for a, b in pairs]
        else:
            ok = [(vals[a][t] > vals[b][t]) == (rew[a] > rew[b]) for a, b in pairs if lens[a] > t and lens[b] > t]
        acc = np.mean(ok) if ok else np.nan
        rows.append({"t": t, "acc": acc, "n": len(ok), "ci": 1.96 * np.sqrt(acc * (1 - acc) / max(len(ok), 1))})
    return pd.DataFrame(rows)

data_sets = {"T=1": load_shards(EVAL, "bon"), "T=0.7": load_shards(FEEDBACK, "tbon")}
data_sets["T=1"] = data_sets["T=1"][data_sets["T=1"]["sample"] < 32].reset_index(drop=True)
for lab, d in data_sets.items():
    assert d.pid.nunique() == len(PIDS) and (d.groupby("pid").size() == 32).all(), f"{lab}: не все сэмплы"

f_pa = f"{RES}/pair_acc_2x2.csv"
if not os.path.exists(f_pa):
    scorers = {"scorer T=1": load_scorer(SCORER_OLD, GEN_DEVICE), "scorer T=0.7": load_scorer(SCORER_NEW, AUX_DEVICE)}
    pid2p = test.set_index("pid").prompt_ids

    def pa_job(sn, lab):
        d = data_sets[lab]
        vals = values_all(scorers[sn], [pid2p[p] for p in d.pid], d.response_ids.tolist(), f"{sn} on {lab}")
        pairs = [(a, a + 1) for a in range(0, len(d), 2) if abs(d.reward[a] - d.reward[a + 1]) > PAIR_MIN_GAP]
        return pair_acc_table(vals, d.response_ids.map(len).values, d.reward.values, pairs).assign(scorer=sn, data=lab)

    # по одному scorer'у на GPU одновременно
    with ThreadPoolExecutor(2) as ex:
        tabs = [f.result() for f in [ex.submit(lambda sn=sn: pd.concat([pa_job(sn, lab) for lab in data_sets]), ) for sn in scorers]]
    pd.concat(tabs).to_csv(f_pa, index=False)
    del scorers
    torch.cuda.empty_cache()

pa = pd.read_csv(f_pa)
pa["cell"] = pa.acc.map("{:.3f}".format) + " ± " + pa.ci.map("{:.3f}".format)
print(pa.pivot_table(index="t", columns=["data", "scorer"], values="cell", aggfunc="first", sort=False).to_string())
print("\nчисло пар:", pa[pa.scorer == "scorer T=1"].pivot(index="t", columns="data", values="n").reindex(pa.t.unique()).T.to_string())

## 4. CD при T=0.7 с новым scorer'ом

Те же 3 конфигурации, промпты, батчи и сиды, что в `04`. Поэтому кандидаты первого блока, как правило, совпадают со старым
прогоном, и разница новый − старый меряется точнее. Reward считается Skywork 0.6B.

In [ ]:
def run_cd(pol, sc, prompt_ids, M, K, gen_kw):
    n = len(prompt_ids)
    resp, fin, n_blocks, gen_tok = [[] for _ in range(n)], [False] * n, [0] * n, [0] * n
    done = [False] * n
    while not all(done):
        act = [i for i in range(n) if not done[i]]
        seqs = [prompt_ids[i] + resp[i] for i in act]
        max_new = min(M, max(MAX_NEW_TOKENS - len(resp[i]) for i in act))
        cands = sample(pol, seqs, K, max_new, gen_kw)
        flat, meta = [], []
        for j, i in enumerate(act):
            rem = MAX_NEW_TOKENS - len(resp[i])
            for k in range(K):
                ids, f = cands[j * K + k]
                if len(ids) > rem:
                    ids, f = ids[:rem], False
                gen_tok[i] += len(ids)
                flat.append(seqs[j] + ids); meta.append((ids, f))
        v = values(sc, flat)
        for j, i in enumerate(act):
            ids, f = meta[j * K + int(np.argmax(v[j * K:(j + 1) * K]))]
            resp[i] += ids; n_blocks[i] += 1; fin[i] = f
            done[i] = f or len(resp[i]) >= MAX_NEW_TOKENS
    return resp, fin, n_blocks, gen_tok

todo = [(ci_, M, K) for ci_, (M, K) in enumerate(TEMP_CD) if any(not os.path.exists(res_path(f"ncd_M{M}_K{K}", s)) for s in range(n_shards))]
if todo:
    policy = load_policy(GEN_DEVICE)
    sc_new = load_scorer(SCORER_NEW, AUX_DEVICE)
    rm = load_rm(AUX_DEVICE)
    for ci_, M, K in todo:
        name = f"ncd_M{M}_K{K}"
        pbar = tqdm(total=len(test), desc=f"T=0.7 {name}", unit="prompt")
        for s in range(n_shards):
            sh = test.iloc[s * SHARD_SIZE:(s + 1) * SHARD_SIZE]
            if os.path.exists(res_path(name, s)):
                pbar.update(len(sh)); continue
            torch.manual_seed(SEED_04 * 100_003 + 60_000 + 1000 * ci_ + s)  # как в 04
            resp, fin, nb, gt = run_cd(policy, sc_new, sh.prompt_ids.tolist(), M, K, GEN_KW_T07)
            df = pd.DataFrame({"pid": sh.pid.values, "response_ids": resp, "finished": fin, "n_blocks": nb, "gen_tokens": gt,
                               "response": [decode(r) for r in resp]})
            df["reward"] = rewards(rm, sh.messages.tolist(), df.response.tolist())
            save_parquet(df, res_path(name, s))
            pbar.update(len(sh))
        pbar.close()
    del policy, sc_new, rm
    torch.cuda.empty_cache()

## Итог: старый и новый scorer против best-of-K при T=0.7

- `K_eq`, `Δ_tok`: как в `03` и `04`, относительно best-of-K при T=0.7.
- `Δ new−old`: разница reward CD с новым и старым scorer'ом на тех же промптах, парный бутстреп.

**Как читать:**
- новый scorer даёт значимый `Δ_tok` > 0, то есть CD снова лучше best-of-K: верна гипотеза (б), CD нужно обучать под ту
  температуру, с которой он работает;
- `Δ new−old` около нуля и в разделе 3 оба scorer'а одинаково точны на ответах при T=0.7: верна гипотеза (а);
- промежуточный случай: новый scorer лучше старого, но best-of-K не догоняет. Тогда сдвиг распределения есть, но объясняет
  только часть проигрыша.

In [ ]:
def ks_upto(kmax):
    return [2 ** i for i in range(int(math.log2(kmax)) + 1)]

def bon_curve(SEL, VAL, Ks):
    n = SEL.shape[0]
    out = []
    for K in Ks:
        pick = SEL.reshape(n, -1, K).argmax(2)
        out.append(np.take_along_axis(VAL.reshape(n, -1, K), pick[..., None], 2)[..., 0].mean(1))
    return np.stack(out, 1)

def k_eq(c, curve, Ks):
    return 2 ** np.interp(c, np.maximum.accumulate(curve), np.log2(Ks))

def ci(x):
    lo, hi = np.percentile(x, [2.5, 97.5])
    return f"[{lo:.2f}; {hi:.2f}]"

BOOT = np.random.default_rng(0).integers(0, len(PIDS), (N_BOOT, len(PIDS)))

def compare(curve, c, Ks, bon_tok, cd_tok):
    cb, bb = c[BOOT].mean(1), curve[BOOT].mean(1)
    ke = k_eq(c.mean(), curve.mean(0), Ks)
    j = int(np.searchsorted(bon_tok, cd_tok))
    over, j = j >= len(Ks), min(j, len(Ks) - 1)
    d = cb - bb[:, j]
    return {"K_eq": ("≥" if ke >= Ks[-1] else "") + f"{ke:.1f}", "K_eq_ci": ci([k_eq(x, y, Ks) for x, y in zip(cb, bb)]),
            "vs_tok": (">" if over else "") + f"best-of-{Ks[j]}", "Δ_tok": round(c.mean() - curve[:, j].mean(), 2),
            "Δ_tok_ci": ci(d), "P(Δ≤0)": round(float((d <= 0).mean()), 3)}

def length_row(method, df):
    return {"method": method, "resp_len": df.response_ids.map(len).mean(), "finished": df.finished.mean(),
            "reward": df.reward.mean(), "reward_finished": df[df.finished].reward.mean()}

tb = data_sets["T=0.7"]
Kt = ks_upto(32)
R = tb.reward.values.reshape(len(PIDS), 32)
curve = bon_curve(R, R, Kt)
toks = np.array(Kt) * tb.response_ids.map(len).mean()
print("best-of-K при T=0.7:", dict(zip(Kt, curve.mean(0).round(2))), "\n")

rows, diffs = [], []
for M, K in TEMP_CD:
    old, new = load_shards(FEEDBACK, f"tcd_M{M}_K{K}"), load_shards(RES, f"ncd_M{M}_K{K}")
    if new is None or new.pid.nunique() != len(PIDS):
        print(f"M={M} K={K}: новый CD посчитан не на всех промптах, пропускаю"); continue
    for lab, d in [("T=1", old), ("T=0.7", new)]:
        rows.append({"scorer": lab, **length_row(f"CD M={M} K={K}", d), "gen_tok": d.gen_tokens.mean(),
                     **compare(curve, d.reward.values, Kt, toks, d.gen_tokens.mean())})
    dd = new.reward.values - old.reward.values
    db = dd[BOOT].mean(1)
    diffs.append({"method": f"CD M={M} K={K}", "Δ new−old": round(dd.mean(), 2), "ci": ci(db), "P(Δ≤0)": round(float((db <= 0).mean()), 3)})

summary = pd.DataFrame(rows)
summary.to_csv(f"{RES}/summary_t07.csv", index=False)
print(summary[["method", "scorer", "reward", "gen_tok", "resp_len", "finished", "K_eq", "K_eq_ci", "vs_tok", "Δ_tok", "Δ_tok_ci", "P(Δ≤0)"]]
      .round(2).to_string(index=False))
print("\nНовый scorer против старого:")
print(pd.DataFrame(diffs).to_string(index=False))

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(toks, curve.mean(0), "o-", color="black", label="best-of-K, T=0.7")
for lab, col in [("T=1", "tab:gray"), ("T=0.7", "tab:red")]:
    d = summary[summary.scorer == lab]
    ax.plot(d.gen_tok, d.reward, "^", color=col, ms=9, label=f"CD, scorer обучен при {lab}")
    for _, r in d.iterrows():
        ax.annotate(r.method[3:], (r.gen_tok, r.reward), fontsize=7, xytext=(4, -10), textcoords="offset points")
ax.set_xscale("log", base=2); ax.set_xlabel("сгенерировано токенов на промпт"); ax.set_ylabel("reward (Skywork 0.6B)")
ax.set_title("Генерация при T=0.7, top-p=0.9"); ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{RES}/t07.png", dpi=150); plt.show()

## Сохранение в Kaggle Dataset

Два приватных датасета: `cd-hh-scorer-t07` (веса нового scorer'а, ~1 ГБ) и `cd-hh-t07` (роллауты при T=0.7, ответы CD,
таблицы, график). Нужен секрет `KAGGLE_API_TOKEN`.

In [ ]:
!pip install -q -U kaggle

import subprocess
from kaggle_secrets import UserSecretsClient
os.environ["KAGGLE_API_TOKEN"] = UserSecretsClient().get_secret("KAGGLE_API_TOKEN")

USERNAME = "karimvafin"
SUFFIX = "-debug" if DEBUG else ""

def kaggle_cli(*args):
    r = subprocess.run(["kaggle", *args], capture_output=True, text=True)
    return r.returncode, (r.stdout + r.stderr).strip()

def upload_dir(src, slug, message="update"):
    # Kaggle CLI без --dir-mode пропускает подпапки, поэтому загружаем плоские папки
    stage = f"{WORK}/upload_{slug}"
    shutil.rmtree(stage, ignore_errors=True)
    shutil.copytree(src, stage)
    ds_id = f"{USERNAME}/{slug}"
    json.dump({"title": slug, "id": ds_id, "licenses": [{"name": "CC0-1.0"}]}, open(f"{stage}/dataset-metadata.json", "w"))
    exists = kaggle_cli("datasets", "status", ds_id)[0] == 0
    code, out = kaggle_cli("datasets", "version", "-p", stage, "-m", message) if exists else kaggle_cli("datasets", "create", "-p", stage)
    if code != 0 and not exists:
        print(out[-2000:], "\n→ create не удался, пробую version")
        code, out = kaggle_cli("datasets", "version", "-p", stage, "-m", message)
        exists = True
    print(out[-2000:])
    assert code == 0, "загрузка не удалась, см. вывод выше; файлы остались в " + stage
    shutil.rmtree(stage, ignore_errors=True)
    print(f"{'новая версия' if exists else 'создан'}: https://www.kaggle.com/datasets/{ds_id}")

if SCORER_NEW == FINAL:  # обучен в этой сессии
    upload_dir(FINAL, "cd-hh-scorer-t07" + SUFFIX, message="scorer trained on T=0.7 rollouts")
upload_dir(RES, "cd-hh-t07" + SUFFIX, message="T=0.7 scorer experiment")